In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

plt.style.use('seaborn-v0_8-whitegrid')

FAST_TRACK = True

PARAMS = (
    dict(widths=[64, 128], n_layers=8,  n_seeds=3)
    if FAST_TRACK else
    dict(widths=[64, 128, 256, 512], n_layers=30, n_seeds=10)
)

sigma_w_critical = 1.0 / np.sqrt(0.456)
print(f'sigma_w* = {sigma_w_critical:.6f}')
print(f'Mode: {"FAST-TRACK" if FAST_TRACK else "FULL"}')
print(f'Widths:  {PARAMS["widths"]}')
print(f'Layers:  {PARAMS["n_layers"]}')
print(f'Seeds:   {PARAMS["n_seeds"]}')

In [ ]:
def measure_correlation_profile(n, m, sigma_w, seed):
    rng = np.random.default_rng(seed)
    xi  = np.empty(m)
    for k in range(m):
        W  = rng.standard_normal((n, n)) * sigma_w / np.sqrt(n)
        ev = np.clip(np.linalg.eigvalsh((W.T @ W) / n), 1e-10, None)
        xi[k] = 1.0 / np.sqrt(np.mean(1.0 / ev))
    return xi

def fit_exponential(xi):
    k = np.arange(len(xi), dtype=float)
    def exp_model(k, xi_0, k_c):
        return xi_0 * np.exp(-k / k_c)
    try:
        popt, _ = curve_fit(
            exp_model, k, xi,
            p0=[xi[0], max(len(xi) / 3.0, 1.0)],
            bounds=([0.0, 0.1], [np.inf, np.inf]),
            maxfev=10_000,
        )
    except RuntimeError:
        popt = [xi[0], float(len(xi))]
    xi_pred = exp_model(k, *popt)
    ss_res  = ((xi - xi_pred) ** 2).sum()
    ss_tot  = ((xi - xi.mean()) ** 2).sum()
    r2      = 1.0 - ss_res / max(ss_tot, 1e-12)
    return {'xi_0': popt[0], 'k_c': popt[1], 'r2': r2, 'xi_pred': xi_pred}

In [ ]:
all_results = {}

for width in PARAMS['widths']:
    print(f'Width N = {width}')
    seed_results = []
    for seed in range(PARAMS['n_seeds']):
        xi  = measure_correlation_profile(
            width, PARAMS['n_layers'], sigma_w_critical, seed * 1000 + width
        )
        fit = fit_exponential(xi)
        fit['xi_values'] = xi
        seed_results.append(fit)
        print(f'  seed {seed}:  xi_0={fit["xi_0"]:.4f}  '
              f'k_c={fit["k_c"]:.3f}  R2={fit["r2"]:.4f}')

    r2_arr = np.array([r['r2'] for r in seed_results])
    print(f'  Mean R2 = {r2_arr.mean():.4f} +/- {r2_arr.std():.4f}  '
          f'pass rate = {(r2_arr >= 0.95).mean():.2f}\n')
    all_results[width] = seed_results

In [ ]:
BLUES  = ['#d1e5f0', '#92c5de', '#4393c3', '#2166ac']
k_arr  = np.arange(PARAMS['n_layers'])

fig, axes = plt.subplots(1, len(PARAMS['widths']),
                         figsize=(6 * len(PARAMS['widths']), 5))
if len(PARAMS['widths']) == 1:
    axes = [axes]

for ax, width in zip(axes, PARAMS['widths']):
    for i, result in enumerate(all_results[width]):
        c = BLUES[min(i, len(BLUES) - 1)]
        ax.plot(k_arr, result['xi_values'], 'o-', color=c,
                alpha=0.7, linewidth=1.2, markersize=3, label=f'Seed {i}')
        ax.plot(k_arr, result['xi_pred'], '--', color=c,
                alpha=0.5, linewidth=1.5)
    mean_r2 = np.mean([r['r2'] for r in all_results[width]])
    ax.set_title(f'N={width}  |  Mean R2={mean_r2:.4f}')
    ax.set_xlabel('Layer k')
    ax.set_ylabel('xi(k)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('nb02_h1_exponential_decay.pdf', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
all_r2       = [r['r2'] for results in all_results.values() for r in results]
overall_pass = np.mean(np.array(all_r2) >= 0.95)

print(f'Overall mean R2:    {np.mean(all_r2):.4f}')
print(f'Overall pass rate:  {overall_pass:.2f}')
print(f'H1 VALIDATED:       {overall_pass >= 0.90}')